## Non Linear Transformer

In [3]:
import pandas as pd
import numpy as np

import torch

### CUDA framework & Device Info

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Compute device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"Compute capability: {props.major}.{props.minor}")
    print(f"VRAM available: {props.total_memory / 1e9:.1f} GB")

PyTorch version: 2.12.0+cu130
Compute device: cuda
GPU: NVIDIA GeForce RTX 2080 Super with Max-Q Design
CUDA version: 13.0
cuDNN version: 92000
Compute capability: 7.5
VRAM available: 8.6 GB


### Seed & Configuration

### Additional CUDA settings

Performance settings for modern GPUs under CUDA 13.0. TF32 accelerates float32 matmul and convolution on Ampere and later at the cost of slightly reduced precision relative to full fp32. Uncomment the following cell as per requirements.

In [ ]:
# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True
# torch.backends.cudnn.benchmark = True
# torch.set_float32_matmul_precision("high")

### Annual Lag Appending for K1 Characteristics

For each such characteristic, the rank-normalised current observation is augmented with five annual lags at months $\{12, 24, 36, 48, 60\}$ to get six observations per K1 characteristic. The resulting flat input vector $\bar{x}_{i,t} \in \mathbb{R}^{K_0 + 6K_1}$ is constructed exclusively from information available at or before month $t$.

In [ ]:
LAG_MONTHS = [0, 12, 24, 36, 48, 60]


def append_annual_lags( panel: pd.DataFrame, k1_cols: list[str], date_col: str = "date", firm_col: str = "gvkey") -> pd.DataFrame: 

    panel = panel.sort_values([firm_col, date_col]).copy()
    for col in k1_cols:
        for lag in LAG_MONTHS[1:]:
            lag_col = f"{col}_lag{lag}"
            shifted = (panel[[firm_col, date_col, col]].copy().assign(**{date_col: lambda df: df[date_col] + pd.DateOffset(months=lag)}).rename(columns={col: lag_col})
            )
            panel = panel.merge(shifted, on=[firm_col, date_col], how="left")
    return panel


def build_flat_input_vector(panel: pd.DataFrame, k0_cols: list[str], k1_cols: list[str], date_col: str = "date", 
                            firm_col: str = "permno") -> tuple[np.ndarray, list[str], pd.DataFrame]:

    ordered = list(k0_cols)
    for col in k1_cols:
        ordered.append(col)
        for lag in LAG_MONTHS[1:]:
            ordered.append(f"{col}_lag{lag}")
    available = [c for c in ordered if c in panel.columns]
    X = panel[available].to_numpy(dtype=np.float32)
    meta = panel[[firm_col, date_col]].reset_index(drop=True)
    expected = len(k0_cols) + 6 * len(k1_cols)
    print(
        f"Flat input vector assembled: {X.shape[0]:,} observations, "
        f"{X.shape[1]} features (expected K0 + 6*K1 = {expected})."
    )
    return X, available, meta